# Aircraft Maintenance Manual - Knowledge Graph Pipeline

This notebook uses `SimpleKGPipeline` from `neo4j-graphrag` to transform the A320-200 Maintenance Manual into a searchable knowledge graph with a single pipeline call. The pipeline automates:

1. **Chunking** - Splitting the document into searchable pieces
2. **Embedding** - Generating vector representations for semantic search
3. **Entity extraction** - Using an LLM to identify operating limits from the text
4. **Entity resolution** - Deduplicating extracted entities across chunks

The pipeline produces two things: a **searchable chunk layer** with Document and Chunk nodes carrying embeddings, which powers semantic search in the rest of this lab, and **extracted `ExtractedLimit` entities**, the thresholds the LLM read out of the manual's prose. The chunk layer is the foundation; the extracted entities are what turn unstructured text into graph-connected knowledge.

Lab 2 already loaded 20 canonical `OperatingLimit` nodes from `silver_operating_limits`, transcribed from these same manuals by hand. Extraction does not add to that set and does not overwrite it. It writes a separate label, `ExtractedLimit`, under the same naming convention, so you can put the LLM's reading of a limit next to the hand transcription of the same limit and see where the two agree. That comparison is the point, and it only works while the two populations are distinguishable.

After the pipeline runs, we create indexes for retrieval and cross-link the extracted knowledge to the aircraft topology from Lab 2.

**Prerequisites:**
- Complete **Lab 2** (Databricks ETL) to load the aircraft topology graph (Aircraft, System, Component nodes)
- Running in a Databricks notebook environment

**Learning Objectives:**
- Understand the Document -> Chunk graph structure for semantic search
- Configure and run SimpleKGPipeline for automated chunking, embedding, and entity extraction
- Create vector and fulltext indexes for retrieval
- Cross-link both limit populations to the aircraft topology (APPLIES_TO, HAS_LIMIT, HAS_EXTRACTED_LIMIT)
- Perform semantic similarity search over maintenance procedures

---

## Why SimpleKGPipeline?

Building a GraphRAG knowledge graph involves several steps: splitting documents into chunks, embedding them, extracting structured entities with an LLM, and resolving duplicate entities. `SimpleKGPipeline` orchestrates all of these in a single pass, producing:

```
(:Document) <-[:FROM_DOCUMENT]- (:Chunk) -[:NEXT_CHUNK]-> (:Chunk)
                                    |
                              (embedding vector)
                              (extracted entities: ExtractedLimit)
```

The `ExtractedLimit` entities are one join point back to the operational graph: each one links to a `Sensor` from Lab 2 via `HAS_EXTRACTED_LIMIT`, alongside the `HAS_LIMIT` edge that carries Lab 2's canonical threshold for the same sensor.

The pipeline reads the maintenance manual, chunks it, uses an LLM to extract `ExtractedLimit` entities such as EGT (Exhaust Gas Temperature) thresholds and vibration limits, and stores everything in Neo4j with embeddings for semantic search. The manuals are written in airline shorthand, and the [workshop glossary](https://neo4j-partners.github.io/databricks-neo4j-workshop/databricks-neo4j-workshop/1.0/glossary.html) expands the terms these chunks come back full of.

## Section 1: Configuration

Nothing to type. Lab 1's `02_credentials_and_cypher.ipynb` stored your Neo4j credentials in a Databricks secret scope named `fleet-ops-<your user>`, and this cell reads them back. Notebooks 02 and 03 and Lab 5 read the same scope the same way.

Databricks redacts secret values in notebook output, so the URI prints as `[REDACTED]`. That is expected, not a bug. The value is intact in Python and the Neo4j connection works.

If this cell stops with an error about a missing scope, go back and run Lab 1 notebook 02. It is what creates the scope.


In [ ]:
# ============================================
# CONFIGURATION - read the credentials Lab 1 stored
# ============================================
from data_utils import read_neo4j_secrets, secret_scope_name

SECRET_SCOPE = secret_scope_name(spark)
credentials = read_neo4j_secrets(dbutils, SECRET_SCOPE)
NEO4J_URI = credentials["uri"]
NEO4J_USERNAME = credentials["username"]
NEO4J_PASSWORD = credentials["password"]
NEO4J_DATABASE = credentials["database"]

# Unity Catalog Volume path (pre-configured by workshop admin)
DATA_PATH = "/Volumes/databricks-neo4j-workshop/aircraft/raw_data"

print(f"Secret scope: {SECRET_SCOPE}")
print(f"Neo4j URI: {NEO4J_URI}")
print(f"Neo4j database: {NEO4J_DATABASE}")
print(f"Data Path: {DATA_PATH}")


## Setup

Import required modules and configure the environment.

In [ ]:
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index

from data_utils import (
    Neo4jConnection, VolumeDataLoader, get_embedder, get_llm,
    run_pipeline, EMBEDDING_DIMENSIONS
)

## Maintenance Manual Data

We'll load the A320-200 Maintenance and Troubleshooting Manual from the Unity Catalog Volume. This file was uploaded during lab setup along with the aircraft CSV data.

**Volume Path:** `/Volumes/databricks-neo4j-workshop/aircraft/raw_data/MAINTENANCE_A320.md`

This comprehensive document includes:

- **Aircraft specifications** for the SkyWays A320-200 fleet (5 aircraft)
- **System architecture** covering Engines (V2500-A1), Avionics, and Hydraulics
- **Troubleshooting procedures** with fault codes and decision trees
- **Operating limits** and scheduled maintenance tasks

This realistic maintenance manual will allow semantic search queries like:
- "How do I troubleshoot engine vibration?"
- "What are the EGT limits during takeoff?"
- "What causes hydraulic pressure loss?"

In [ ]:
# Load text from the maintenance manual in Unity Catalog Volume
loader = VolumeDataLoader("MAINTENANCE_A320.md", volume_path=DATA_PATH)
MANUAL_TEXT = loader.text

# Document metadata
DOCUMENT_ID = "AMM-A320-2024-001"
AIRCRAFT_TYPE = "A320-200"

metadata = loader.get_metadata()
print(f"Loaded: {metadata['name']}")
print(f"From Volume: {metadata['volume']}")
print(f"Size: {metadata['size']:,} characters")
print(f"\nFirst 500 characters:")
print(f"{MANUAL_TEXT[:500]}...")

## Connect to Neo4j

Create a connection to your Neo4j database. This should already contain the aircraft topology from Lab 2.

In [ ]:
neo4j = Neo4jConnection(uri=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE).verify()
driver = neo4j.driver

# Show existing graph statistics
neo4j.get_graph_stats()

## Clear Previous Data (Optional)

Run this for a clean start. It removes what this notebook creates: Document nodes, Chunk nodes, and the `ExtractedLimit` entities the extraction step produces. `SimpleKGPipeline` creates chunks with no dedup key, so a second run without this leaves you with two copies of every chunk and duplicate embeddings in every search result.

It leaves the Lab 2 graph alone, `OperatingLimit` included. That label now means the 20 canonical rows Lab 2 loaded from `silver_operating_limits` and nothing else, so the delete needs no filter on it: it simply is not in the label list.

In [ ]:
neo4j.clear_enrichment()

## Initialize LLM and Embedder

Set up the Large Language Model (LLM) and embedding model for the pipeline.

- **LLM**: Uses Databricks Foundation Model APIs (Claude Sonnet 5) for entity extraction
- **Embedder**: Uses Databricks Foundation Model APIs (BGE-large) for chunk embeddings

In [ ]:
llm = get_llm()
embedder = get_embedder()

print(f"LLM initialized: {llm.model_id}")
print(f"Embedder initialized: {embedder.model_id}")

---

# Part 1: SimpleKGPipeline

The `SimpleKGPipeline` from `neo4j-graphrag` handles the full transformation from raw text to a queryable knowledge graph in a single call. It:

1. **Splits** the document into chunks (800 characters with 100-character overlap)
2. **Embeds** each chunk using the BGE embedding model (1024-dimensional vectors)
3. **Extracts** `ExtractedLimit` entities from each chunk using the LLM
4. **Resolves** duplicate entities (the same limit mentioned across multiple chunks becomes one node)
5. **Stores** everything in Neo4j: Document, Chunk nodes with embeddings, and ExtractedLimit entities

### Chunking Trade-offs

Larger chunks give the LLM more context during entity extraction, improving its ability to resolve references like "the engine" to a specific component. Smaller chunks produce more precise retrieval results. A moderate chunk size (800 characters) with overlap (100 characters) at boundaries is a good starting point.

### Entity Extraction Schema

The pipeline is configured with a schema that tells the LLM what to extract. We define a single entity type:

- **ExtractedLimit** - Aircraft operating parameter thresholds as the LLM read them out of the manual: EGT limits, vibration thresholds, and the like
  - Properties: `name`, `parameterName`, `unit`, `regime`, `minValue`, `maxValue`, `aircraftType`

The label is deliberately not `OperatingLimit`. That one belongs to Lab 2's canonical rows. Names still follow the `<parameterName> - <aircraftType>` convention the CSV uses, so `EGT - A320-200` extracted here sits next to `EGT - A320-200` transcribed there and the two compare directly.

A custom extraction prompt teaches the LLM to distinguish aircraft types (A320-200) from engine types (V2500) and to use sensor parameter names (EGT, Vibration, N1Speed) that match the operational graph. `N1Speed` is fan speed, as a percentage of the engine's rated speed.

### Context-Aware Chunking

A `ContextPrependingSplitter` prepends a two-line header to every chunk:

```
[DOCUMENT CONTEXT] Aircraft Type: A320-200 | Title: A320-200 Maintenance Manual
[SECTION] 4. Engine Troubleshooting Procedures > 4.2 Vibration Exceedance
```

The first line carries the aircraft type. Without it, chunks deep in engine-specific sections lack that context and the LLM confuses engine designations for aircraft types.

The second line names the chunk's section, read from the markdown heading stack at the chunk's position in the manual. Fixed-size splitting cuts at character counts, not at headings, so a chunk that opens mid-section otherwise arrives with no clue where it came from. In this manual roughly half the chunk boundaries land inside a table or an ASCII decision tree. The 800-character window is usually wide enough to keep a table's header row with its data, but section identity is lost either way: a chunk holding the rows `| 2 | Perform visual inspection of exhaust nozzle | ... |` reads as generic maintenance steps until the header says they belong to `4.1 Overheat Condition (EGT Exceedance) > Troubleshooting Procedure`.

Because the header is part of the stored chunk text, it is embedded too, so the section name contributes to semantic search as well as to extraction. A chunk that straddles a heading is attributed to the section it begins in.

## The Extraction Schema

The prose above describes the schema, but the schema itself lives in `data_utils.py`. The `run_pipeline()` helper calls `build_extraction_schema()` and hands the result to `SimpleKGPipeline(schema=...)`, which is why you don't see it in the pipeline cell below.

Run the cell to view the actual `ExtractedLimit` entity definition the pipeline uses. Keeping it in `data_utils.py` means there is a single source of truth: this cell always reflects what the pipeline really runs, including the label it writes.

In [ ]:
from data_utils import build_extraction_schema

# The exact schema SimpleKGPipeline uses for entity extraction.
# This is defined in data_utils.py and passed into the pipeline by run_pipeline().
extraction_schema = build_extraction_schema()

for node in extraction_schema.node_types:
    print(f"Entity: {node.label}")
    print(f"  {node.description}\n")
    for prop in node.properties:
        print(f"  - {prop.name} ({prop.type}): {prop.description}")

# A closed schema: the LLM extracts ONLY this entity type, no relationships,
# and may not invent additional node types.
print(f"\nRelationship types: {extraction_schema.relationship_types or '(none)'}")
print(f"Additional node types allowed: {extraction_schema.additional_node_types}")

## The Extraction Prompt

The schema tells the LLM *what* to extract. A custom prompt tells it *how*. The `EXTRACTION_PROMPT` (also in `data_utils.py`) is what teaches the LLM the rules that make extraction reliable:

- Use the aircraft type from the `[DOCUMENT CONTEXT]` line on every entity.
- Distinguish the airframe (A320-200) from the engine model (V2500), which appears throughout the text but is a component, not the aircraft type.
- Name each entity `<parameterName> - <aircraftType>` so the same limit on different aircraft does not get merged during entity resolution.
- Only extract when the text contains specific numeric limits or ranges.

Print it to see exactly what the LLM is instructed to do.

In [ ]:
from data_utils import EXTRACTION_PROMPT

# The custom prompt that teaches the LLM how to extract ExtractedLimit entities.
# {schema}, {examples}, and {text} are filled in by the pipeline at runtime.
print(EXTRACTION_PROMPT)

## Run the Pipeline

Process the full maintenance manual. The pipeline will chunk the text, generate embeddings, and extract operating limit entities.

> **Note:** This cell takes several minutes. The manual splits into roughly 40 chunks and the LLM reads each one in turn. `SimpleKGPipeline` has a `max_concurrency` setting, but it only helps when the LLM client is genuinely async. `DatabricksLLM.ainvoke` in `data_utils.py` wraps a synchronous `mlflow.deployments` call, so the chunks are processed one after another and the wall-clock time is roughly the number of chunks times the per-chunk response time.

In [ ]:
run_pipeline(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    llm=llm,
    embedder=embedder,
    text=MANUAL_TEXT,
    document_metadata={
        "documentId": DOCUMENT_ID,
        "aircraftType": AIRCRAFT_TYPE,
        "title": "A320-200 Maintenance and Troubleshooting Manual",
        "type": "maintenance_manual",
    },
    context=f"[DOCUMENT CONTEXT] Aircraft Type: {AIRCRAFT_TYPE} | Title: A320-200 Maintenance Manual\n\n",
)

## Inspect the Knowledge Graph

The pipeline created Document, Chunk, and ExtractedLimit nodes. Let's verify what was built.

In [ ]:
# Updated graph statistics
neo4j.get_graph_stats()

# Document-Chunk structure
records, _, _ = driver.execute_query("""
    MATCH (d:Document)
    OPTIONAL MATCH (d)<-[:FROM_DOCUMENT]-(c:Chunk)
    RETURN d.documentId AS document_id, d.title AS title, count(c) AS chunks
""", database_=NEO4J_DATABASE)
print("\n=== Document-Chunk Structure ===")
for record in records:
    print(f"Document: {record['document_id']}")
    print(f"  Title: {record['title']}")
    print(f"  Chunks: {record['chunks']}")

# Chunk chain sample
records, _, _ = driver.execute_query("""
    MATCH (c:Chunk)
    WHERE c.index IS NOT NULL
    OPTIONAL MATCH (c)-[:NEXT_CHUNK]->(next:Chunk)
    RETURN c.index AS idx,
           substring(c.text, 0, 80) AS text,
           next.index AS next_idx
    ORDER BY c.index
    LIMIT 5
""", database_=NEO4J_DATABASE)
print("\n=== Chunk Chain (first 5) ===")
for record in records:
    next_str = f" -> Chunk {record['next_idx']}" if record['next_idx'] is not None else " (end)"
    print(f"Chunk {record['idx']}: \"{record['text']}...\"{next_str}")

## Inspect Extracted Entities

The LLM extracted `ExtractedLimit` entities from the maintenance manual. These represent specific operating thresholds like maximum EGT temperatures, vibration limits, and fuel flow ranges.

Entity resolution has already deduplicated these: the same limit mentioned across multiple chunks appears as a single node.

`MATCH (el:ExtractedLimit)` is the whole selector. Lab 2's canonical limits carry a different label, so there is nothing to filter out.

In [ ]:
# ExtractedLimit is extraction output only. Lab 2's canonical limits are
# OperatingLimit nodes and never appear in this result.
records, _, _ = driver.execute_query("""
    MATCH (el:ExtractedLimit)
    RETURN el.name AS name, el.parameterName AS param,
           el.aircraftType AS aircraft, el.unit AS unit,
           el.maxValue AS max_value, el.regime AS regime
    ORDER BY el.name
""", database_=NEO4J_DATABASE)
print(f"Extracted {len(records)} ExtractedLimit entities:\n")
for r in records:
    regime = f" ({r['regime']})" if r['regime'] else ""
    unit = r['unit'] or ''
    max_val = r['max_value'] or 'N/A'
    print(f"  {r['name']}: max={max_val} {unit}{regime} [{r['aircraft']}]")

---

# Part 2: Create Indexes for Retrieval

The pipeline stored embeddings on each Chunk node. Now we create indexes so retrieval queries can efficiently search them:

- **Vector index** - For semantic similarity search (cosine similarity over 1024-dimensional embeddings, where a higher score means a closer match)
- **Fulltext index** - For keyword-based search (enables `HybridRetriever` in later notebooks)

In [ ]:
INDEX_NAME = "maintenanceChunkEmbeddings"

create_vector_index(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    name=INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=EMBEDDING_DIMENSIONS,
    similarity_fn="cosine"
)
print(f"Created vector index: {INDEX_NAME} ({EMBEDDING_DIMENSIONS} dimensions, cosine similarity)")

In [ ]:
FULLTEXT_INDEX_NAME = "maintenanceChunkText"

create_fulltext_index(
    driver=driver,
    neo4j_database=NEO4J_DATABASE,
    name=FULLTEXT_INDEX_NAME,
    label="Chunk",
    node_properties=["text"]
)
print(f"Created fulltext index: {FULLTEXT_INDEX_NAME}")

---

# Part 3: Cross-Linking to Aircraft Topology

The pipeline created Document, Chunk, and `ExtractedLimit` nodes, and none of them is connected to the aircraft topology from Lab 2 yet. Three cross-links bridge these worlds:

1. **`Document -[:APPLIES_TO]-> Aircraft`**. The Document node carries an `aircraftType` of `A320-200`, which matches `Aircraft.model`. This lets retrieval queries traverse from a matched chunk through its Document to the specific aircraft it applies to. Nine aircraft in the Lab 2 fleet are A320-200s, so this one Document links to all nine.

2. **`Sensor -[:HAS_LIMIT]-> OperatingLimit`**. Lab 2 loaded its 20 canonical limits as reference data and left them unattached. Each one carries a `parameterName` such as `EGT` that matches `Sensor.type`, and an `aircraftType` that matches `Aircraft.model`. This is where each limit gets wired to the sensors it judges.

3. **`Sensor -[:HAS_EXTRACTED_LIMIT]-> ExtractedLimit`**. The same match run against the extraction output, under its own relationship type.

The second relationship type is what this cell is really teaching. `HAS_LIMIT` answers the question "what is the threshold", so it has to reach the hand-transcribed number, and it is the edge the retrievers in notebooks 02 and 03 follow. `HAS_EXTRACTED_LIMIT` answers a different question: "what did the LLM read out of the manual for this sensor". Keep them separate and one sensor reaches both numbers, so you can compare them. Reuse `HAS_LIMIT` for extraction output and a retriever has no way to tell which of the two numbers it was handed.

Both limit edges hang off the same `Sensor`. They are siblings, not a chain:

```
Chunk -[:FROM_DOCUMENT]-> Document -[:APPLIES_TO]-> Aircraft
                                                        |
                                                 [:HAS_SYSTEM]
                                                        v
                                                     System
                                                        |
                                                 [:HAS_SENSOR]
                                                        v
                                                     Sensor
                                                    /        \
                                        [:HAS_LIMIT]          [:HAS_EXTRACTED_LIMIT]
                                              v                        v
                                       OperatingLimit            ExtractedLimit
                                     (Lab 2, transcribed)      (this lab, LLM-read)
```

## First, Coerce the Extracted Numeric Bounds

Before anything links to an `ExtractedLimit`, its bounds have to be numbers.

The LLM read these limits out of prose, so it handed back text. `minValue` and
`maxValue` arrive as strings such as `"695"`, not as numbers. The extraction
schema does declare a type for every property, but that declaration is a hint
inside the prompt. `neo4j-graphrag` never validates or converts what the model
emits, so the declared type is a request, not a guarantee.

This applies to the extracted set alone. Lab 2's canonical `OperatingLimit`
bounds arrive already typed: the Spark load casts `minValue` and `maxValue` to
Double on the way in, so there is nothing to clean there.

Text where a number belongs fails quietly in Cypher. Comparing a float reading
against a string limit, `r.value > el.maxValue`, evaluates to null rather than
raising, so a limit-exceedance query returns zero rows and reads as a clean
answer. No error, no result, no signal.

Clean at the boundary, right after extraction. The cell below converts a bound
to a float only when it parses as one. A value the model wrote as a range or
with its unit attached, `"620-680"` or `"695 °C"`, stays exactly as it is and
is reported back to you, because a visible wrong type beats silent data loss.
Re-running is safe: a bound that is already numeric is left alone. `minValue`
is absent on single-bound limits such as a vibration ceiling, which is correct
data rather than a defect.

In [ ]:
# The LLM returns every extracted property as text. Convert the numeric bounds
# to float so a comparison against a sensor reading evaluates. Only the
# extracted set needs this; Lab 2's OperatingLimit bounds are already Double.
BOUND_PROPERTIES = ("minValue", "maxValue")

# Classify first, so the report describes what extraction actually produced.
CLASSIFY_BOUND = """
    MATCH (el:ExtractedLimit)
    WITH el.{bound} AS raw
    WITH raw, CASE
        WHEN raw IS NULL THEN 'null or absent'
        WHEN NOT (valueType(raw) STARTS WITH 'STRING') THEN 'already numeric'
        WHEN toFloatOrNull(raw) IS NOT NULL THEN 'parses as a number'
        ELSE 'could not parse'
    END AS status
    RETURN status, count(*) AS nodes, collect(DISTINCT raw)[..10] AS samples
"""

# Three guards, one per rule: skip a missing bound, skip one that is already
# numeric, and skip anything toFloatOrNull cannot read.
COERCE_BOUND = """
    MATCH (el:ExtractedLimit)
    WHERE el.{bound} IS NOT NULL
      AND valueType(el.{bound}) STARTS WITH 'STRING'
      AND toFloatOrNull(el.{bound}) IS NOT NULL
    SET el.{bound} = toFloatOrNull(el.{bound})
    RETURN count(*) AS coerced
"""

totals, _, _ = driver.execute_query(
    "MATCH (el:ExtractedLimit) RETURN count(*) AS nodes", database_=NEO4J_DATABASE
)
node_count = totals[0]["nodes"]
print(f"ExtractedLimit nodes: {node_count}\n")

if node_count == 0:
    print("Nothing to coerce: the pipeline extracted no ExtractedLimit entities.")
    print("Notebook 02's operating-limit retriever still works. It reads Lab 2's")
    print("canonical OperatingLimit nodes, which do not come from extraction.")
else:
    for bound in BOUND_PROPERTIES:
        found, _, _ = driver.execute_query(
            CLASSIFY_BOUND.format(bound=bound), database_=NEO4J_DATABASE
        )
        counts = {r["status"]: r["nodes"] for r in found}
        unparsed = next(
            (r["samples"] for r in found if r["status"] == "could not parse"), []
        )

        written, _, _ = driver.execute_query(
            COERCE_BOUND.format(bound=bound), database_=NEO4J_DATABASE
        )

        print(f"{bound}")
        print(f"  coerced to float: {written[0]['coerced']}")
        print(f"  already numeric:  {counts.get('already numeric', 0)}")
        print(f"  null or absent:   {counts.get('null or absent', 0)}")
        print(f"  could not parse:  {counts.get('could not parse', 0)}")
        if unparsed:
            print(f"    left as text:   {unparsed}")
        print()

In [ ]:
# Document -[:APPLIES_TO]-> Aircraft
records, _, _ = driver.execute_query("""
    MATCH (d:Document) WHERE d.aircraftType IS NOT NULL
    MATCH (a:Aircraft {model: d.aircraftType})
    MERGE (d)-[:APPLIES_TO]->(a)
    RETURN d.documentId AS doc, a.model AS aircraft, count(*) AS count
""", database_=NEO4J_DATABASE)
print("Document -[:APPLIES_TO]-> Aircraft:")
for r in records:
    print(f"  {r['doc']} -> {r['aircraft']}")

# Sensor -[:HAS_LIMIT]-> OperatingLimit, the canonical limits Lab 2 loaded.
# MERGE, so running this cell again creates nothing new.
records, _, _ = driver.execute_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(sys:System)-[:HAS_SENSOR]->(s:Sensor)
    MATCH (ol:OperatingLimit {parameterName: s.type, aircraftType: a.model})
    MERGE (s)-[:HAS_LIMIT]->(ol)
    RETURN ol.name AS limit_name, count(*) AS sensors
    ORDER BY limit_name
""", database_=NEO4J_DATABASE)
print("\nSensor -[:HAS_LIMIT]-> OperatingLimit, transcribed by hand in Lab 2:")
for r in records:
    print(f"  {r['sensors']:>3} sensors -> {r['limit_name']}")

# Sensor -[:HAS_EXTRACTED_LIMIT]-> ExtractedLimit, the LLM's reading of the same
# thresholds. Its own relationship type, so the two sets stay comparable and a
# retriever always knows which number it asked for.
records, _, _ = driver.execute_query("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(sys:System)-[:HAS_SENSOR]->(s:Sensor)
    MATCH (el:ExtractedLimit {parameterName: s.type, aircraftType: a.model})
    MERGE (s)-[:HAS_EXTRACTED_LIMIT]->(el)
    RETURN el.name AS limit_name, count(*) AS sensors
    ORDER BY limit_name
""", database_=NEO4J_DATABASE)
print("\nSensor -[:HAS_EXTRACTED_LIMIT]-> ExtractedLimit, read by the LLM:")
for r in records:
    print(f"  {r['sensors']:>3} sensors -> {r['limit_name']}")
if not records:
    print("  No matches. This depends on which limits the LLM extracted.")

In [ ]:
# Verify the traversal Chunk -> Document -> Aircraft -> System -> Sensor, then
# count each limit population separately at the end of it.
records, _, _ = driver.execute_query("""
    MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d:Document)-[:APPLIES_TO]->(a:Aircraft)
    WITH a, count(c) AS chunks
    OPTIONAL MATCH (a)-[:HAS_SYSTEM]->(sys:System)-[:HAS_SENSOR]->(s:Sensor)
    OPTIONAL MATCH (s)-[:HAS_LIMIT]->(ol:OperatingLimit)
    OPTIONAL MATCH (s)-[:HAS_EXTRACTED_LIMIT]->(el:ExtractedLimit)
    RETURN a.tail_number AS aircraft, a.model AS model, chunks,
           count(DISTINCT ol) AS canonical,
           count(DISTINCT el) AS extracted
    ORDER BY a.tail_number
""", database_=NEO4J_DATABASE)
print("Full graph traversal (Chunk -> Document -> Aircraft -> ... -> limits):")
print(f"{'Aircraft':<12} {'Model':<12} {'Chunks':<8} {'Canonical':<11} {'Extracted':<10}")
print("-" * 55)
for r in records:
    print(f"{r['aircraft']:<12} {r['model']:<12} {r['chunks']:<8} {r['canonical']:<11} {r['extracted']:<10}")
print("\nCanonical counts the limits Lab 2 transcribed by hand. Extracted counts")
print("what the LLM read from the manual. A gap between them is the measurement.")

---

# Part 4: Test Semantic Search

With embeddings stored and the vector index created, we can now search for chunks that are semantically similar to a query. The search:
1. Converts the query to an embedding vector
2. Finds chunks with similar embedding vectors using cosine similarity
3. Returns results ranked by similarity score, highest first

In [ ]:
def vector_search(driver, embedder, query: str, top_k: int = 3):
    """Search for chunks similar to the query."""
    query_embedding = embedder.embed_query(query)

    records, _, _ = driver.execute_query("""
        CALL db.index.vector.queryNodes($index_name, $top_k, $embedding)
        YIELD node, score
        RETURN node.text as text, node.index as idx, score
    """, index_name=INDEX_NAME, top_k=top_k, embedding=query_embedding, database_=NEO4J_DATABASE)

    return records

# Test search with a maintenance query
query = "How do I troubleshoot engine vibration?"
print(f"Query: \"{query}\"\n")
print("=" * 70)

results = vector_search(driver, embedder, query)
for i, record in enumerate(results):
    print(f"\n[{i+1}] Score: {record['score']:.4f} (Chunk {record['idx']})")
    print(f"    {record['text'][:250]}...")

## Compare Different Queries

Try different maintenance queries to see how semantic search finds relevant procedures even with different wording.

In [ ]:
queries = [
    "What are the EGT limits during takeoff?",
    "How to detect bearing wear in the engine?",
    "What causes hydraulic pressure loss?",
    "When should I replace the fuel filter?"
]

for query in queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 60)
    results = vector_search(driver, embedder, query, top_k=1)
    if results:
        record = results[0]
        print(f"Best match (score: {record['score']:.4f}):")
        print(f"  {record['text'][:200]}...")

## Summary

In this notebook, you built the foundation for semantic search over aircraft maintenance documentation:

1. **Document-Chunk Structure**: loaded the A320-200 Maintenance Manual as a Document node connected to text Chunks via `FROM_DOCUMENT` relationships, with sequential `NEXT_CHUNK` links
2. **Knowledge Graph Construction**: used `SimpleKGPipeline` to chunk text, generate embeddings, and extract entities as `ExtractedLimit` nodes in a single pipeline run, kept under their own label so Lab 2's canonical `OperatingLimit` rows stay untouched
3. **Vector and Fulltext Indexes**: created both `maintenanceChunkEmbeddings` for vectors and `maintenanceChunkText` for fulltext
4. **Cross-Links to Topology**: connected Documents to Aircraft via `APPLIES_TO`, Lab 2's canonical limits to their Sensors via `HAS_LIMIT`, and this lab's extracted limits to the same Sensors via `HAS_EXTRACTED_LIMIT`, bridging unstructured text to the structured graph from Lab 2
5. **Search Verification**: validated that vector similarity search returns relevant results. The fulltext index was created but not queried here; it is exercised by the hybrid retrievers in notebook 03

**Next Steps:**
- Continue to [GraphRAG Retrievers](02_graphrag_retrievers.ipynb) to build retrieval strategies that combine vector search with graph traversal
- **Optional:** [Hybrid Retrievers](03_hybrid_retrievers.ipynb) - Explore hybrid search that combines vector similarity with keyword matching

In [ ]:
# Cleanup
neo4j.close()